In [53]:
from datetime import date, datetime, timedelta

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_ai_model.validation import flights_df_comparison
import polars as pl
DAY_MARGIN = timedelta(minutes=15)

test_date = date(2026,3,1)
df_gold = get_bts_flights_for_day(test_date)
# df_test = get_adsbx_flights_for_day(test_date)
df_test = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')

expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
min_time = datetime(test_date.year, test_date.month, test_date.day) + DAY_MARGIN
max_time = datetime(test_date.year, test_date.month, test_date.day) + timedelta(days=1) - DAY_MARGIN
expr2 = (pl.col("takeoff_time") > min_time) & (pl.col("landing_time") < max_time)
expr = expr & expr2
df_gold = df_gold.filter(expr)
df_test = df_test.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
df,stats = flights_df_comparison(df_gold, df_test)

226 overlapping flights found for 2026-03-01

--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 4385
Predicted flights: 13996  |  Gold flights: 15515
Correctly matched: 13814  |  Predicted unmatched: 182  |  Gold flights missed: 1701
Precision: 98.70%  |  Recall: 89.04%  |  F1: 93.62%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 13774/13783 matched (99.93%)
landing_airport_ident: 13735/13737 matched (99.99%)
overall: 27509/27520 matched (99.96%)


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_7891/1884531950.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))


In [54]:
COLUMNS = ["icao", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df2", "takeoff_airport_ident_df2", "landing_time_df2","landing_airport_ident_df2", "match_status"]
df = df.select(COLUMNS)

In [55]:
df.filter(pl.col("match_status") == "df1_only")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a027ca""",2026-03-01 01:07:00,"""KCPR""",2026-03-01 01:51:00,"""KDEN""",null,null,null,null,"""df1_only"""
"""a05f31""",2026-03-01 01:43:00,"""KBNA""",2026-03-01 03:26:00,"""KLGA""",null,null,null,null,"""df1_only"""
"""a0b59d""",2026-03-01 12:24:00,"""KIDA""",2026-03-01 12:57:00,"""KSLC""",null,null,null,null,"""df1_only"""
"""a0b81d""",2026-03-01 01:30:00,"""KPHX""",2026-03-01 04:25:00,"""KORD""",null,null,null,null,"""df1_only"""
"""a0d90c""",2026-03-01 21:55:00,"""KRIC""",2026-03-01 22:44:00,"""KCLT""",null,null,null,null,"""df1_only"""
…,…,…,…,…,…,…,…,…,…
"""ad97dd""",2026-03-01 16:38:00,"""KPIB""",2026-03-01 17:40:00,"""KIAH""",null,null,null,null,"""df1_only"""
"""ad9db8""",2026-03-01 16:33:00,"""KORD""",2026-03-01 17:19:00,"""KDTW""",null,null,null,null,"""df1_only"""
"""ada6b9""",2026-03-01 12:24:00,"""KSLN""",2026-03-01 13:33:00,"""KDEN""",null,null,null,null,"""df1_only"""


In [56]:
df.filter(pl.col("icao") == "a0b59d").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a0b59d""",2026-03-01 00:50:00,"""KAUS""",2026-03-01 03:38:00,"""KSLC""",2026-03-01 00:49:43.665,"""KAUS""",2026-03-01 03:38:13.849,"""KSLC""","""matched"""
"""a0b59d""",2026-03-01 12:24:00,"""KIDA""",2026-03-01 12:57:00,"""KSLC""",null,null,null,null,"""df1_only"""
"""a0b59d""",2026-03-01 16:13:00,"""KSLC""",2026-03-01 17:48:00,"""KPDX""",2026-03-01 16:08:36.994,"""KSLC""",2026-03-01 17:48:11.812,"""KPDX""","""matched"""
"""a0b59d""",2026-03-01 18:47:00,"""KPDX""",2026-03-01 20:10:00,"""KSLC""",2026-03-01 18:47:43.245,"""KPDX""",2026-03-01 20:10:58.403,"""KSLC""","""matched"""
"""a0b59d""",2026-03-01 21:22:00,"""KSLC""",2026-03-01 23:32:00,"""KAUS""",2026-03-01 21:17:04.017,"""KSLC""",2026-03-01 23:32:58.658,"""KAUS""","""matched"""
